In [1]:
# 连接数据库

import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("数据库连接成功")

数据库连接成功


In [5]:
# 快速备份

import sqlite3

src = "db/bible.db"
backup = "db/bible_backup.sqlite"

with sqlite3.connect(src) as src_conn:
    with sqlite3.connect(backup) as bak_conn:
        src_conn.backup(bak_conn)

print("SQLite 备份完成")

SQLite 备份完成


In [5]:
# 快速恢复

import sqlite3

backup = "db/bible_backup.sqlite"
target = "db/bible.db"

with sqlite3.connect(backup) as src_conn:
    with sqlite3.connect(target) as dst_conn:
        src_conn.backup(dst_conn)

print("数据库恢复完成")

数据库恢复完成


In [3]:
# 填充指定章（手动）的 tokens 表，从 verse 表获取数据

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # ✅ 填充 word_seq 和 align_ID（只改这里）
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"

        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=3
    )

📖 处理章节：Gen 3
  需要处理的 verse 数：24
✅ 插入 token 数：1456
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：674
🎉 本章 token 构建完成


In [ ]:
# mp3 -> wav 
# 增量更新，要改成指定章，为了工作流

import os
import subprocess
import time

MP3_DIR = "outputs/mp3"
WAV_DIR = "outputs/wav"

def batch_mp3_to_wav(mp3_dir: str, wav_dir: str, sample_rate: str = "16000"):
    if not os.path.isdir(mp3_dir):
        raise FileNotFoundError(f"❌ 输入目录不存在：{mp3_dir}")

    os.makedirs(wav_dir, exist_ok=True)

    # 以文件名（不含后缀）做差集
    mp3_files = {
        f[:-4] for f in os.listdir(mp3_dir)
        if f.lower().endswith(".mp3")
        and os.path.isfile(os.path.join(mp3_dir, f))
    }
    wav_files = {
        f[:-4] for f in os.listdir(wav_dir)
        if f.lower().endswith(".wav")
        and os.path.isfile(os.path.join(wav_dir, f))
    }

    to_convert = sorted(mp3_files - wav_files)
    skipped = len(mp3_files & wav_files)

    if skipped:
        print(f"⏭️ 已存在跳过：{skipped} 个")
    if not to_convert:
        print("✅ 全部已同步，无需转换")
        return

    print(f"🎬 待转换 {len(to_convert)} 个\n")

    total = len(to_convert)
    t_start = time.time()

    for i, name in enumerate(to_convert, 1):
        mp3_path = os.path.join(mp3_dir, f"{name}.mp3")
        wav_path = os.path.join(wav_dir, f"{name}.wav")

        cmd = [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", mp3_path,
            "-ar", sample_rate,
            "-ac", "1",
            wav_path
        ]

        s = time.time()
        subprocess.run(cmd, check=True)
        cost = time.time() - s

        print(f"[{i}/{total}] ✅ {name}.wav  ({cost:.2f}s)")

    print(f"\n🏁 完成！共转换 {total} 个，总耗时 {time.time() - t_start:.2f}s")


if __name__ == "__main__":
    batch_mp3_to_wav(MP3_DIR, WAV_DIR, sample_rate="16000")

In [ ]:
# 使用 plaintext 和 wav，输出对应章的 TextGrid
# 增量更新，要改成指定章，为了工作流

import os
import shutil
import subprocess

# ===== 目录 =====
WAV_DIR = "outputs/wav"
TXT_DIR = "outputs/plaintext"
CORPUS_DIR = "corpus"
ALIGN_OUT = "outputs/forcealign"

# ===== MFA 模型路径（✅ 关键修复）=====
dict_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/dictionary/english_us_arpa.dict"
)
acoustic_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/acoustic/english_us_arpa.zip"
)

# ===== 注入 aligner 环境 =====
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()
aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

def prepare_and_align():
    os.makedirs(CORPUS_DIR, exist_ok=True)
    os.makedirs(ALIGN_OUT, exist_ok=True)

    # 已对齐结果
    aligned = {
        f[:-9] for f in os.listdir(ALIGN_OUT)
        if f.endswith(".TextGrid")
    }

    wavs = {f[:-4] for f in os.listdir(WAV_DIR) if f.endswith(".wav")}
    txts = {f[:-4] for f in os.listdir(TXT_DIR) if f.endswith(".txt")}

    ready = wavs & txts
    to_align = sorted(ready - aligned)

    if not to_align:
        print("✅ 所有样本已完成强制对齐")
        return

    print(f"🎯 待对齐：{len(to_align)} 个")

    for name in to_align:
        shutil.copy(
            os.path.join(WAV_DIR, f"{name}.wav"),
            os.path.join(CORPUS_DIR, f"{name}.wav")
        )
        shutil.copy(
            os.path.join(TXT_DIR, f"{name}.txt"),
            os.path.join(CORPUS_DIR, f"{name}.txt")
        )

    cmd = [
        "mfa", "align", CORPUS_DIR,
        dict_path,
        acoustic_path,
        ALIGN_OUT,
        "--clean", "--overwrite"
    ]

    print("🚀 开始强制对齐...\n")
    subprocess.run(cmd, check=True)
    print("\n🏁 强制对齐完成")

if __name__ == "__main__":
    prepare_and_align()

In [6]:
# 使用 TextGrid ，指定章 填充 timestamps 表

import re
import sqlite3
import os

DB_PATH = "db/bible.db"
TEXTGRID_DIR = "outputs/forcealign"

# ========= 交互输入 =========
book_abbr = input("请输入书卷简称（如 01_Gen）：").strip()
chapter = int(input("请输入章数（如 1）：").strip())

# =========================

# ✅ TextGrid 文件名规则
filename = f"{book_abbr}_{chapter:03d}_en.TextGrid"
filepath = os.path.join(TEXTGRID_DIR, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"❌ 找不到文件: {filepath}")

with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()

# ======================
# 1️⃣ 只提取 item [1] 区块
# ======================
item1_pattern = re.compile(
    r"item\s*\[1\]:(.*?)(?=\n\s*item\s*\[\d+\]:|\Z)",
    re.DOTALL
)

m = item1_pattern.search(content)
if not m:
    raise ValueError("❌ 未找到 item [1]，请检查 TextGrid 结构")

item1_content = m.group(1)

# ======================
# 2️⃣ 解析 intervals
# ======================
interval_pattern = re.compile(
    r"intervals\s*\[\d+\]:\s*\n"
    r"\s*xmin\s*=\s*([\d.]+)\s*\n"
    r"\s*xmax\s*=\s*([\d.]+)\s*\n"
    r'\s*text\s*=\s*"([^"]*)"',
    re.MULTILINE
)

matches = interval_pattern.findall(item1_content)

# ======================
# 3️⃣ 写入数据库
# ======================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='timestamps'"
)
if cursor.fetchone() is None:
    raise RuntimeError("❌ timestamps 表不存在，请先建表")

word_seq = 0
for xmin, xmax, text in matches:
    if text.strip() == "":
        continue

    word_seq += 1

    start_ms = int(round(float(xmin) * 1000, 0))
    end_ms = int(round(float(xmax) * 1000, 0))

    # ✅ align_id：下划线 + 补零
    align_id = f"{book_abbr}_{chapter:03d}__{word_seq:04d}"

    cursor.execute(
        """
        INSERT INTO timestamps (
            id, chapter, word_seq, word, start_time, end_time
        ) VALUES (?, ?, ?, ?, ?, ?)
        """,
        (align_id, chapter, word_seq, text, start_ms, end_ms)
    )

    print(f"{align_id:20s} | {text:10s} | {start_ms:>6} | {end_ms:>6}")

conn.commit()
conn.close()

print(f"\n✅ 共插入 {word_seq} 条记录（仅 item [1]，已忽略空 text）")

请输入书卷简称（如 01_Gen）：01_Gen
请输入章数（如 1）：3
01_Gen_003__0001     | now        |    140 |    440
01_Gen_003__0002     | the        |    440 |    580
01_Gen_003__0003     | serpent    |    580 |   1110
01_Gen_003__0004     | was        |   1110 |   1330
01_Gen_003__0005     | more       |   1330 |   1550
01_Gen_003__0006     | crafty     |   1550 |   2130
01_Gen_003__0007     | than       |   2130 |   2340
01_Gen_003__0008     | any        |   2400 |   2700
01_Gen_003__0009     | other      |   2700 |   2940
01_Gen_003__0010     | wild       |   2940 |   3350
01_Gen_003__0011     | animal     |   3350 |   3890
01_Gen_003__0012     | that       |   4040 |   4230
01_Gen_003__0013     | the        |   4230 |   4350
01_Gen_003__0014     | lord       |   4350 |   4740
01_Gen_003__0015     | god        |   4740 |   5120
01_Gen_003__0016     | had        |   5120 |   5370
01_Gen_003__0017     | made       |   5370 |   5850
01_Gen_003__0018     | he         |   6430 |   6560
01_Gen_003__0019     | sai